In [1]:
import pandas as pd

In [ ]:
Res_activity = pd.read_csv('./Results_hubs_activity_stats_shuffle_corrected.tsv', sep='\t')
GRN = pd.read_csv("./Data/Network_GRN_HIVE_curated.txt", sep='\t')
degree_GRN = pd.read_csv('./TDA/Degree_CGRN.tsv', sep='\t')
degree_GRN.rename(columns={'Gene_Name': 'Gene'}, inplace=True)
mapper_res = pd.read_csv('./TDA/Mapper_node_df.tsv', sep='\t')
mapper_res.rename(columns={'Value': 'TDA_Cluster'}, inplace=True)
TF_res = Res_activity[Res_activity['OLN'].isin(GRN['tf.name'].unique())]
DEA_padj = pd.read_csv('./DEA/Merge_padj_groups_all.tsv', sep='\t', index_col=0)
DEA_padj.reset_index(inplace=True)
DEA_padj['index'] = DEA_padj['index'].str.split(':').str.get(1).str.split('.').str.get(0)

In [64]:
merge_degree = pd.merge(TF_res, degree_GRN, left_on='OLN', right_on='Gene', how='left')
merge_degree.drop(columns=['Gene'], inplace=True)

In [65]:
group_cluster = {1: [0,2,3,5],
                  2: [7],
                  3: [9,18,13,15,17,11],
                  4: [19],
                  0: [1,4,6,8,10,12,14,16]}
cluster_to_group = {
    cluster: group
    for group, clusters in group_cluster.items()
    for cluster in clusters
}
mapper_res['TDA_Group'] = mapper_res['TDA_Cluster'].map(cluster_to_group)
Mapper_format = (
    mapper_res
    .groupby('Gene', as_index=False)
    .agg({
        'TDA_Cluster': lambda x: sorted(set(x)),
        'TDA_Group': lambda x: sorted(set(x))
    })
)

In [66]:
merge_tda = pd.merge(merge_degree, Mapper_format, left_on='OLN', right_on='Gene', how='left')
merge_tda.drop(columns=['Gene'], inplace=True)

In [70]:
final_merge = pd.merge(merge_tda, DEA_padj, left_on='OLN', right_on='index', how='left')
final_merge.drop(columns=['index'], inplace=True)
final_merge.rename(columns={'gene.description': 'Description','gene.family':'TF Family'}, inplace=True)

In [73]:
columns = ['OLN', 'Gene_Name', 'Description', 'TF Family', 'In_Degree', 'Out_Degree', 'Total_Degree', 'TDA_Cluster', 'TDA_Group',
           'stats_Pinfestans','stats_Cfulvum', 'stats_PSTVd_S23','stats_PSTVd_M','stats_Bcinerea', 'stats_Mincognita_7dpi', 'stats_Mincognita_14dpi', 
           'padj_Pinfestans','padj_Cfulvum', 'padj_PSTVd_S23','padj_PSTVd_M', 'padj_Bcinerea', 'padj_Mincognita_7dpi', 'padj_Mincognita_14dpi', 'Groups_DEA', 
           'Pinfestans_acts','Cfulvum_acts', 'PSTVd_S23_acts','PSTVd_M_acts', 'Bcinerea_acts', 'Mincognita_7dpi_acts', 'Mincognita_14dpi_acts',
           'Pinfestans_pval','Cfulvum_pval', 'PSTVd_S23_pval','PSTVd_M_pval', 'Bcinerea_pval', 'Mincognita_7dpi_pval', 'Mincognita_14dpi_pval', 'Groups', 'Groups_corrected']

In [76]:
final_merge = final_merge[columns]

In [77]:
final_merge.to_csv('./Supplementary_Table_4_Details_TFs.tsv', sep='\t', index=False)